# v3 fine-tuning — 레이 도메인 데이터 추가 (Kaggle / T4)

`best_v2_ft.pt`를 출발점으로, **레이 영상 라벨 데이터**를 추가해 번호판(+미러·로고)을 실전 도메인에 맞춤.

**핵심: 레이 데이터 oversampling**
- 레이는 26장뿐이라 그냥 합치면 전체의 0.6%로 묻힘 → **repeat로 여러 번 복제**해 비중을 올림
- 기존 4종 리허설은 400→250으로 축소해 상대 비중 조정
- 레이는 이미 전역 클래스(0~4)로 재매핑됨 → 리맵 건너뜀(`keep:"asis"`)

**실행 전 확인**
- Settings → Accelerator = GPU T4, Internet = On
- Add Input **2개**:
  1. 기존 데이터셋 (`.bin` 6개 + best_v2_ft.pt 혹은 best.pt)
  2. 새 `ray-data` (`ray_data.bin` + `best_v2_ft.pt`)

In [ ]:
!pip install ultralytics -q

In [ ]:
# ============================================================
# 0. 입력 경로 자동 탐색 (input 폴더 2개를 각각 찾음)
# ============================================================
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
print("input 아래 목록:")
for p in INPUT_ROOT.iterdir():
    print(" -", p.name, "->", [f.name for f in p.iterdir()][:8])

# 출발점 모델: best_v2_ft.pt 우선, 없으면 best.pt
BEST = None
for name in ["best_v2_ft.pt", "best.pt"]:
    hits = list(INPUT_ROOT.rglob(name))
    if hits:
        BEST = hits[0]; break
assert BEST is not None, "출발점 모델(best_v2_ft.pt/best.pt)을 못 찾음"
print("\n출발점 모델:", BEST)

# 기존 .bin 들이 있는 폴더 (handle.bin 기준으로 탐색)
BASE_DIR = None
for p in INPUT_ROOT.rglob("handle.bin"):
    BASE_DIR = p.parent; break
assert BASE_DIR is not None, "기존 데이터셋(handle.bin 등)을 못 찾음"
print("기존 데이터 폴더:", BASE_DIR)

# 레이 데이터 (ray_data.bin)
RAY_BIN = None
for p in INPUT_ROOT.rglob("ray_data.bin"):
    RAY_BIN = p; break
assert RAY_BIN is not None, "ray_data.bin 을 못 찾음 - ray-data 데이터셋을 Add Input 했는지 확인"
print("레이 데이터:", RAY_BIN)

In [ ]:
# ============================================================
# 1. 설정
# ============================================================
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

WORK   = Path("/kaggle/working/_ft_src")
MERGED = Path("/kaggle/working/_ft_merged")

REHEARSAL_PER_CLASS = 250   # 기존 4종: 클래스당 최대 (v2의 400에서 축소 → 레이 비중↑)
RAY_REPEAT = 15             # ★ 레이 데이터 복제 횟수 (26장 x15 ≈ 390장, 비중 ↑)
VAL_RATIO = 0.15
SEED = 42
EPOCHS = 20
IMGSZ = 640
BATCH = 32

GLOBAL_NAMES = ["handle", "mirror", "plate", "logo", "fuel_cap"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# keep: "all" | "name:X" | "asis"(이미 전역클래스 → 그대로)
# full: True=전량, False=리허설(REHEARSAL_PER_CLASS만)
# repeat: 학습셋 복제 횟수 (oversampling)
SOURCES = [
    {"key": "handle",    "file": "handle.bin",    "keep": "all",          "full": False, "repeat": 1, "dir": BASE_DIR},
    {"key": "mirror",    "file": "mirror.bin",    "keep": "name:perfect", "full": False, "repeat": 1, "dir": BASE_DIR},
    {"key": "plate_old", "file": "plate_old.bin", "keep": "all",          "full": True,  "repeat": 1, "dir": BASE_DIR},
    {"key": "plate_new", "file": "plate_new.bin", "keep": "all",          "full": True,  "repeat": 1, "dir": BASE_DIR},
    {"key": "logo",      "file": "logo.bin",      "keep": "all",          "full": False, "repeat": 1, "dir": BASE_DIR},
    {"key": "fuel_cap",  "file": "fuel_cap.bin",  "keep": "name:Fuel-Cap","full": False, "repeat": 1, "dir": BASE_DIR},
    # ★ 레이: 이미 재매핑됨(asis), 전량, 복제 15배
    {"key": "ray",       "file": "ray_data.bin",  "keep": "asis",         "full": True,  "repeat": RAY_REPEAT, "dir": RAY_BIN.parent},
]

In [ ]:
# ============================================================
# 2. .bin 압축 해제
# ============================================================
import shutil, zipfile

if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)

for s in SOURCES:
    src = s["dir"] / s["file"]
    assert src.exists(), f"파일 없음: {src}"
    with zipfile.ZipFile(src) as z:
        z.extractall(WORK / s["key"])
    print(f"  {s['key']:10s} <- {src}")

In [ ]:
# ============================================================
# 3. 라벨 정리 + 전역 클래스 리맵 (레이는 asis라 그대로)
# ============================================================
import yaml

def find_data_yaml(root):
    # ray_data.bin은 안에 ray_data/ 폴더가 한 겹 더 있을 수 있음 → 재귀 탐색
    hits = list(root.rglob("data.yaml"))
    return hits[0] if hits else None

def load_names(root):
    y = find_data_yaml(root)
    return yaml.safe_load(open(y))["names"] if y else []

def collect_pairs(root):
    pairs = []
    for lbl in root.rglob("*.txt"):
        if lbl.parent.name != "labels": continue
        img_dir = lbl.parent.parent / "images"
        for ext in IMG_EXT:
            img = img_dir / (lbl.stem + ext)
            if img.exists():
                pairs.append((img, lbl)); break
    return pairs

def clean_and_remap(s):
    root = WORK / s["key"]
    keep = s["keep"]
    if keep == "asis":
        # 이미 전역 클래스(0~4). 리맵 없이 그대로 수집.
        kept = collect_pairs(root)
        print(f"  [{s['key']}] asis(재매핑됨) -> {len(kept)}장 그대로")
        return kept
    names = load_names(root)
    # gid 결정: 이 소스의 전역 id (asis 아닌 것들은 키로 매핑)
    gid_map = {"handle":0,"mirror":1,"plate_old":2,"plate_new":2,"logo":3,"fuel_cap":4}
    gid = gid_map[s["key"]]
    if keep == "all":
        keep_ids = set(range(len(names)))
    else:
        cname = keep.split("name:",1)[1]
        keep_ids = {names.index(cname)} if cname in names else set(range(len(names)))
        if cname not in names:
            print(f"  ⚠️ [{s['key']}] '{cname}' 없음 -> 전체 유지. names={names}")
    print(f"  [{s['key']}] names={names} | 유지 {sorted(keep_ids)} -> 전역 {gid}({GLOBAL_NAMES[gid]})")
    kept = []
    for img, lbl in collect_pairs(root):
        out = []
        for line in open(lbl):
            p = line.split()
            if p and int(p[0]) in keep_ids:
                p[0] = str(gid); out.append(" ".join(p))
        if out:
            open(lbl,"w").write("\n".join(out)+"\n"); kept.append((img,lbl))
    return kept

per_source = {s["key"]: clean_and_remap(s) for s in SOURCES}
print("\n유지 이미지 수:", {k: len(v) for k,v in per_source.items()})

In [ ]:
# ============================================================
# 4. 병합 (레이는 repeat배 복제 = oversampling)
# ============================================================
import random

if MERGED.exists(): shutil.rmtree(MERGED)
for split in ["train","valid"]:
    (MERGED/split/"images").mkdir(parents=True, exist_ok=True)
    (MERGED/split/"labels").mkdir(parents=True, exist_ok=True)

def copy_pairs(pairs, split, key, tag=""):
    for img, lbl in pairs:
        stem = f"{key}{tag}__{img.stem}"
        shutil.copy(img, MERGED/split/"images"/(stem+img.suffix))
        shutil.copy(lbl, MERGED/split/"labels"/(stem+".txt"))

rng = random.Random(SEED)
summary = {}
for s in SOURCES:
    pairs = per_source[s["key"]][:]
    rng.shuffle(pairs)
    if not s["full"]:
        pairs = pairs[:REHEARSAL_PER_CLASS]
    n_val = max(1, int(len(pairs)*VAL_RATIO)) if pairs else 0
    val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
    # valid는 원본 그대로(복제 안 함 - 평가 왜곡 방지)
    copy_pairs(val_pairs, "valid", s["key"])
    # train은 repeat배 복제
    for r in range(s["repeat"]):
        copy_pairs(train_pairs, "train", s["key"], tag=f"_r{r}")
    summary[s["key"]] = (len(train_pairs)*s["repeat"], len(val_pairs), s["repeat"])

print("소스별 (train복제후 / valid / repeat):")
for k,(tr,va,rp) in summary.items():
    print(f"  {k:10s} train={tr:5d}  valid={va:4d}  x{rp}")
tot_tr = len(list((MERGED/'train'/'images').iterdir()))
tot_va = len(list((MERGED/'valid'/'images').iterdir()))
print(f"  합계 train={tot_tr}  valid={tot_va}")
ray_share = summary["ray"][0] / tot_tr * 100
print(f"  ★ 레이 train 비중: {ray_share:.1f}%")

cfg = {"train": str(MERGED/"train"/"images"), "val": str(MERGED/"valid"/"images"),
       "nc": len(GLOBAL_NAMES), "names": GLOBAL_NAMES}
yaml.safe_dump(cfg, open(MERGED/"data.yaml","w"), allow_unicode=True)

In [ ]:
# ============================================================
# 5. fine-tuning (best_v2_ft.pt에서 출발)
#    먼저 EPOCHS=1로 시간 확인 권장
# ============================================================
from ultralytics import YOLO

model = YOLO(str(BEST))
model.train(
    data=str(MERGED/"data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, patience=10, seed=SEED,
    project="/kaggle/working/ft_runs", name="carparts_v3", exist_ok=True,
)

In [ ]:
# ============================================================
# 6. 클래스별 평가
# ============================================================
!yolo detect val model=/kaggle/working/ft_runs/carparts_v3/weights/best.pt data=/kaggle/working/_ft_merged/data.yaml device=0

In [ ]:
# ============================================================
# 7. 결과 확인 (Output 패널에서 다운로드 → 로컬 models/best_v3_ray.pt 로 저장)
# ============================================================
w = Path("/kaggle/working/ft_runs/carparts_v3/weights/best.pt")
print("완료:", w, "| 존재:", w.exists(), f"| {w.stat().st_size/1e6:.1f} MB" if w.exists() else "")

---
### 학습 후
- best.pt 다운로드 → 로컬 `models/best_v3_ray.pt`
- `test_video.py`로 레이영상 돌려 **plate 검출 프레임 수** 확인 (v1=13, v2=80 → v3=?)
- 4종이 안 무너졌는지 같이 확인
- 여전히 부족하면 → 레이영상 2·3 추가 라벨링(데이터 확대)이 근본 해결

### 튜닝 포인트
- `RAY_REPEAT`(현재 15): 레이 비중. 4종이 무너지면 낮추고, 레이가 약하면 높임
- `REHEARSAL_PER_CLASS`(현재 250): 기존 4종 양. 4종 성능 흔들리면 올림